# =========================================================
# SECTION 1: TUNE HEURISTIC (CẤP 2)
# =========================================================
# So sánh các bộ trọng số khác nhau của Heuristic vs Random
# -> Chốt bộ weights_best

# =========================================================
# SECTION 2: TUNE ALPHA-BETA (CẤP 3) 
# Sử dụng weights_best từ Section 1 cho hàm evaluation
# =========================================================
# So sánh depth=2 vs depth=4 vs depth=6 đối đầu với Heuristic
# -> Chốt depth_best và move_ordering=True

# =========================================================
# SECTION 3: TUNE MCTS (CẤP 4)
# =========================================================
# So sánh các giá trị C và Rollout Policy đối đầu với AlphaBeta(depth_best)
# -> Chốt C_best, rollout_policy_best

In [ ]:
import sys
import os

# Thêm thư mục gốc của dự án vào sys.path
sys.path.append(os.path.abspath('..'))

# (Hoặc chuyển thư mục làm việc hiện tại ra gốc dự án)
# os.chdir('..')

# Sau đó mới import các thư mục trong dự án
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from othello.game import OthelloGame
from agents.heuristic_agent import HeuristicAgent
from agents.alphabeta_agent import AlphaBetaAgent
from agents.mcts_agent import MCTSAgent

# Cấu hình hiển thị đồ thị cho báo cáo
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150

ModuleNotFoundError: No module named 'othello'

In [ ]:
def run_tournament(agent1, agent2, num_games=20):
    """Cho 2 Agent thi đấu num_games ván (luân phiên đổi quân Đen/Trắng để công bằng).
    
    Trả về: (win_rate_agent1, avg_time_agent1, avg_nodes_agent1)
    """
    a1_wins, a2_wins, draws = 0, 0, 0
    a1_times, a1_nodes = [], []
    
    for game_id in range(num_games):
        game = OthelloGame()
        # Đổi bên mỗi ván
        p1, p2 = (agent1, agent2) if game_id % 2 == 0 else (agent2, agent1)
        agent1.reset()
        agent2.reset()
        
        while not game.is_terminal():
            cur = game.current_player
            current_agent = p1 if cur == 1 else p2
            
            start_t = time.perf_counter()
            action = current_agent.select_action(game)
            elapsed = time.perf_counter() - start_t
            
            if current_agent == agent1:
                a1_times.append(elapsed)
                if hasattr(agent1, "stats"):
                    a1_nodes.append(agent1.stats().get("nodes", 0))
                    
            if action is not None:
                game.step(action)
            else:
                game.pass_turn()
                
        winner = game.winner()
        if winner == 0:
            draws += 1
        elif (winner == 1 and p1 == agent1) or (winner == -1 and p2 == agent1):
            a1_wins += 1
        else:
            a2_wins += 1
            
    win_rate = (a1_wins + 0.5 * draws) / num_games
    avg_time = np.mean(a1_times) if a1_times else 0.0
    avg_nodes = np.mean(a1_nodes) if a1_nodes else 0.0
    
    return win_rate, avg_time, avg_nodes

In [ ]:
print("=== THỬ NGHIỆM 1: TUNE ALPHA-BETA AGENT ===")
baseline = HeuristicAgent()

depths = [2, 4, 6]
ab_results = []

for d in depths:
    # 1. Alpha-Beta đầy đủ
    ab_full = AlphaBetaAgent(depth=d, pruning=True, move_ordering=True)
    wr_full, t_full, n_full = run_tournament(ab_full, baseline, num_games=10)
    ab_results.append({"depth": d, "pruning": "Alpha-Beta", "win_rate": wr_full, "time": t_full, "nodes": n_full})
    
    # 2. Minimax thuần (Tắt cắt tỉa - Ablation)
    if d <= 4: # depth 6 Minimax thuần chạy rất lâu nên bỏ qua
        ab_no_prune = AlphaBetaAgent(depth=d, pruning=False, move_ordering=False)
        wr_no, t_no, n_no = run_tournament(ab_no_prune, baseline, num_games=10)
        ab_results.append({"depth": d, "pruning": "Minimax (No Pruning)", "win_rate": wr_no, "time": t_no, "nodes": n_no})

df_ab = pd.DataFrame(ab_results)
print(df_ab.to_string(index=False))

In [ ]:
print("\n=== THỬ NGHIỆM 2: TUNE MCTS AGENT ===")

# 2.1 Tune hằng số C
c_values = [0.5, 0.707, 1.0, 1.414, 2.0]
mcts_c_results = []

for c in c_values:
    mcts = MCTSAgent(simulations=500, exploration_c=c, rollout_policy="heuristic")
    wr, t, _ = run_tournament(mcts, baseline, num_games=10)
    mcts_c_results.append({"c": c, "win_rate": wr, "time": t})

df_mcts_c = pd.DataFrame(mcts_c_results)

# 2.2 So sánh Random Rollout vs Heuristic Rollout theo Simulations
sim_list = [100, 300, 500, 1000]
rollout_results = []

for pol in ["random", "heuristic"]:
    for sim in sim_list:
        mcts = MCTSAgent(simulations=sim, exploration_c=1.414, rollout_policy=pol)
        wr, t, _ = run_tournament(mcts, baseline, num_games=10)
        rollout_results.append({"policy": pol, "simulations": sim, "win_rate": wr, "time": t})

df_mcts_rollout = pd.DataFrame(rollout_results)

In [ ]:
# 1. Đồ thị So sánh Số nút duyệt: Alpha-Beta vs Minimax thuần (Hiệu quả Cắt tỉa)
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=df_ab, x='depth', y='nodes', hue='pruning', ax=ax, palette='Set2')
ax.set_title('Hình 4.1: So sánh Số nút duyệt giữa Alpha-Beta và Minimax thuần', fontsize=12, fontweight='bold')
ax.set_xlabel('Độ sâu tìm kiếm (Depth)', fontsize=11, fontweight='bold')
ax.set_ylabel('Số nút đã duyệt (Nodes)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_alphabeta_nodes.png', dpi=300)
plt.show()

# 2. Đồ thị Hằng số UCT (C) của MCTS
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df_mcts_c['c'], df_mcts_c['win_rate'], marker='o', color='tab:blue', linewidth=2.5, markersize=8)
ax.set_title('Hình 4.2: Tỷ lệ Thắng MCTS theo Hằng số Khám phá UCT (C)', fontsize=12, fontweight='bold')
ax.set_xlabel('Hằng số UCT (C)', fontsize=11, fontweight='bold')
ax.set_ylabel('Tỷ lệ thắng (Win Rate)', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('fig_mcts_c_tuning.png', dpi=300)
plt.show()

# 3. Đồ thị So sánh Rollout Policy của MCTS
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.lineplot(data=df_mcts_rollout, x='simulations', y='win_rate', hue='policy', style='policy', 
             markers=True, dashes=False, linewidth=2.5, markersize=8, ax=ax)
ax.set_title('Hình 4.3: So sánh Random Rollout và Heuristic Rollout theo Simulations', fontsize=12, fontweight='bold')
ax.set_xlabel('Số lượt mô phỏng (Simulations)', fontsize=11, fontweight='bold')
ax.set_ylabel('Tỷ lệ thắng (Win Rate)', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('fig_mcts_rollout_comparison.png', dpi=300)
plt.show()